## 자기소개서 도우미 챗봇 예제(Gradio)

In [ ]:
# 사전 설치 : pip install fpdf
import os
import gradio as gr
from langchain_community.llms import Ollama
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser # LCEL과 함께 사용할 출력 파서
from fpdf import FPDF

In [ ]:
# Ollama 설정 (Gemma2 모델 사용)
os.environ["OLLAMA_API_BASE"] = "http://localhost:11434"  # Ollama 서버 주소
ollama_model = Ollama(model="gemma2")

In [ ]:
# 다양한 템플릿 설정
TEMPLATES = {
    "취업": "다음 키워드와 예시를 바탕으로, 취업 지원을 위한 자기소개서를 작성하세요.",
    "대학원": "제공된 키워드를 사용하여, 대학원 지원을 위한 자기소개서를 초안 작성하세요.",
    "봉사활동": "주어진 키워드를 활용하여, 봉사활동 경험과 동기를 강조하는 자기소개서를 작성하세요."
}

In [ ]:
# 언어 지원: 한국어, 영어, 일본어
LANGUAGES = {
    "한국어": "Please write the response in Korean.",
    "영어": "Please write the response in English.",
    "일본어": "Please write the response in Japanese."
}

In [ ]:
# 자동 키워드 추천 함수
def recommend_keywords(purpose):
    if purpose == "취업":
        return "책임감, 팀워크, 문제 해결 능력"
    elif purpose == "대학원":
        return "연구 열정, 창의력, 학업 성취도"
    elif purpose == "봉사활동":
        return "사회적 책임감, 희생정신, 리더십"
    else:
        return ""

In [ ]:
# 자기소개서 작성 함수
def generate_statement(purpose, language, keywords, example_sentence=None):
    if purpose not in TEMPLATES:
        return "지원 목적을 올바르게 선택해주세요."
    if language not in LANGUAGES:
        return "언어를 올바르게 선택해주세요."

    # 템플릿 생성
    template = TEMPLATES[purpose] + "\n\nKeywords: {keywords}\n" + LANGUAGES[language]
    if example_sentence:
        template += f"\n\nExample sentence: {example_sentence}"

    prompt = PromptTemplate(input_variables=["keywords"], template=template)

    # ======  LCEL 체인 사용 ======
    # 프롬프트, LLM, 출력 파서를 파이프(|)로 간단하게 연결합니다.
    # StrOutputParser는 LLM의 출력에서 문자열만 깔끔하게 추출해줍니다.
    chain = prompt | ollama_model | StrOutputParser()

    # .invoke()에 입력 변수를 딕셔너리 형태로 전달합니다.
    response = chain.invoke({"keywords": keywords})

    return response

In [ ]:
# PDF 저장 함수
def save_to_pdf(statement, filename="personal_statement.pdf"):
    pdf = FPDF()
    pdf.add_page()  # pdf 문서에 새로운 페이지 추가
    pdf.add_font('MalgunGothic', '', r'C:\Windows\Fonts\malgun.ttf', uni=True)  # '맑은 고딕' 폰트 경로 설정
    pdf.set_font('MalgunGothic', size=12)  # 폰트 설정
    # Mac용 폰트 경로 설정
    # font_path = "/System/Library/Fonts/AppleSDGothicNeo.ttc"
    # pdf.add_font('AppleSDGothic', '', font_path, uni=True)
    # pdf.set_font('AppleSDGothic', size=12)

    pdf.multi_cell(0, 10, statement) # PDF 문서에 텍스트를 추가, 셀의너비(0), 셀의 높이(10)
    pdf.output(filename) # PDF 문서를 파일로 저장
    return f"PDF 저장 완료: {filename}"

In [ ]:
# Gradio 인터페이스
def chatbot_interface(purpose, language, keywords, example_sentence=None, save_pdf=False):
    statement = generate_statement(purpose, language, keywords, example_sentence)
    if save_pdf:
        save_to_pdf(statement)
    return statement

In [ ]:
with gr.Blocks() as demo:
    gr.Markdown("# 📝 다목적 자기소개서 작성 도우미")
    gr.Markdown("키워드와 추천 문장을 활용하여 취업, 대학원, 봉사활동 자기소개서를 생성하고 PDF로 저장하세요!")

    # 입력 영역
    with gr.Row():
        purpose_input = gr.Dropdown(label="지원 목적", choices=["취업", "대학원", "봉사활동"], value="취업")
        language_input = gr.Dropdown(label="언어 선택", choices=["한국어", "영어", "일본어"], value="한국어")

    recommended_keywords = gr.Textbox(label="추천 키워드", interactive=False)
    recommend_btn = gr.Button("키워드 추천")
    recommend_btn.click(recommend_keywords, inputs=[purpose_input], outputs=[recommended_keywords])

    with gr.Row():
        keywords_input = gr.Textbox(label="사용자 키워드 입력", placeholder="예: 책임감, 팀워크, 문제 해결 능력")
        example_sentence_input = gr.Textbox(
            label="추천 문장 (선택 사항)",
            placeholder="예: '저는 도전을 두려워하지 않고 성공적으로 프로젝트를 완수했습니다.'"
        )

    save_pdf_toggle = gr.Checkbox(label="PDF로 저장", value=False)

    # 출력 영역
    output = gr.Textbox(label="작성된 자기소개서", lines=6)
    submit_btn = gr.Button("작성하기")
    submit_btn.click(
        fn=chatbot_interface,
        inputs=[purpose_input, language_input, keywords_input, example_sentence_input, save_pdf_toggle],
        outputs=[output]
    )

In [ ]:
# 실행
demo.launch()

In [ ]:
# 종료(자원회수)
demo.close()